# 特征工程与选择

> 一个好特征，胜过一千个数据点。

**类型:** Build
**语言:** Python
**先修:** Phase 1 (Statistics for ML, Linear Algebra), Phase 2 Lessons 1-7
**时间:** ~90 分钟

## 学习目标

- 实现数值变换（standardization、min-max scaling、log transform、binning），并解释每种方法适合什么时候使用
- 为类别特征构建 one-hot、label 和 target encoding，并识别 target encoding 中的数据泄漏风险
- 从零构建 TF-IDF vectorizer，并解释为什么它在文本分类中优于原始词频计数
- 应用基于过滤器的特征选择（variance threshold、correlation、mutual information）来降低维度

## 要解决的问题

你有一个数据集。你选了一个算法。你训练了它。结果一般。你换了一个更复杂的算法。还是一般。你花了一周调超参数。只提升了一点点。

然后有人把原始数据变换成更好的特征，一个简单的 logistic regression 就超过了你调好的 gradient-boosted ensemble。

这件事非常常见。在传统 ML 中，数据的表示方式比算法选择更重要。一个房价模型如果有 "square footage" 和 "number of bedrooms"，无论学习器多复杂，通常都会胜过一个只拿 "address as a raw string" 当输入的模型。算法只能利用你交给它的东西。

特征工程是把原始数据转换成某种表示，让模型更容易发现模式的过程。特征选择则是丢掉那些只增加噪声、不增加信号的特征。两者合在一起，是传统 ML 中杠杆最高的工作之一。

## 核心概念

### 特征流水线

```mermaid
flowchart LR
    A[Raw Data] --> B[Handle Missing Values]
    B --> C[Numerical Transforms]
    B --> D[Categorical Encoding]
    B --> E[Text Features]
    C --> F[Feature Interactions]
    D --> F
    E --> F
    F --> G[Feature Selection]
    G --> H[Model-Ready Data]
```

### 数值特征

原始数字很少能直接供模型使用。常见变换包括：

**Scaling:** 把特征放到同一范围内，让基于距离的算法（K-Means、KNN、SVM）平等对待所有特征。Min-max scaling 会映射到 [0, 1]。Standardization (z-score) 会映射到 mean=0、std=1。

**Log transform:** 压缩右偏分布（收入、人口、词频）。它会把乘法关系转换成加法关系。

**Binning:** 把连续值转换成类别。当特征和目标之间的关系是非线性但分段阶梯式时很有用（例如年龄组）。

**Polynomial features:** 创建 x^2、x^3、x1*x2 这样的项。它让线性模型能够捕捉非线性关系，代价是特征数量会增加。

### 类别特征

模型需要数字。类别需要编码。

**One-hot encoding:** 为每个类别创建一个二进制列。"color = red/blue/green" 会变成三列：is_red、is_blue、is_green。它适合低基数特征，但类别很多时会让维度爆炸。

**Label encoding:** 把每个类别映射成整数：red=0、blue=1、green=2。它会引入假的顺序关系（模型可能以为 green > blue > red）。只有在基于树的模型按单个取值切分时才比较合适。

**Target encoding:** 用该类别对应目标变量的均值替换每个类别。很强大，但也很危险：数据泄漏风险很高。它必须只在训练数据上计算，再应用到测试数据。

### 文本特征

**Count vectorizer:** 统计每个词在文档中出现了多少次。"the cat sat on the mat" 会变成 {the: 2, cat: 1, sat: 1, on: 1, mat: 1}。

**TF-IDF:** Term Frequency-Inverse Document Frequency。它按词在所有文档中有多独特来加权。像 "the" 这样的常见词权重会很低。稀有且有辨识度的词会得到高权重。

```text
TF(word, doc) = count(word in doc) / total words in doc
IDF(word) = log(total docs / docs containing word)
TF-IDF = TF * IDF
```

### 缺失值

真实数据总会有空洞。常见策略：

- **删除行：** 只在缺失数据很少且随机缺失时使用
- **均值/中位数填补：** 简单，能保留分布形状（中位数对离群值更稳健）
- **众数填补：** 用于类别特征
- **指示列：** 填补前添加一个二进制列 "was_this_missing"。数据缺失这件事本身可能就是有信息的
- **前向/后向填充：** 用于时间序列数据

### 特征交互

有时关系藏在组合里。"Height" 和 "weight" 单独看预测力较弱，而 "BMI = weight / height^2" 会更有预测力。特征交互会放大特征空间，所以要用领域知识挑选合适的交互项。

### 特征选择

特征越多不一定越好。无关特征会增加噪声、拉长训练时间，并可能导致过拟合。

**过滤方法（建模前）：**
- Correlation：移除彼此高度相关的特征（冗余）
- Mutual information：衡量知道某个特征后，目标的不确定性减少了多少
- Variance threshold：移除几乎不变化的特征

**包装方法（基于模型）：**
- L1 regularization (Lasso)：把无关特征的权重推到恰好为零
- Recursive feature elimination：训练，移除最不重要的特征，然后重复

**为什么选择很重要：** 一个有 10 个好特征的模型，通常会胜过一个有 10 个好特征加 90 个噪声特征的模型。噪声特征会给模型机会去记住训练数据中无法泛化的模式。

## 动手实现

### 步骤 1：从零实现数值变换


In [ ]:
import math


def min_max_scale(values):
    min_val = min(values)
    max_val = max(values)
    if max_val == min_val:
        return [0.0] * len(values)
    return [(v - min_val) / (max_val - min_val) for v in values]


def standardize(values):
    n = len(values)
    mean = sum(values) / n
    variance = sum((v - mean) ** 2 for v in values) / n
    std = math.sqrt(variance) if variance > 0 else 1.0
    return [(v - mean) / std for v in values]


def log_transform(values):
    return [math.log(v + 1) for v in values]


def bin_values(values, n_bins=5):
    min_val = min(values)
    max_val = max(values)
    bin_width = (max_val - min_val) / n_bins
    if bin_width == 0:
        return [0] * len(values)
    result = []
    for v in values:
        bin_idx = int((v - min_val) / bin_width)
        bin_idx = min(bin_idx, n_bins - 1)
        result.append(bin_idx)
    return result


def polynomial_features(row, degree=2):
    n = len(row)
    result = list(row)
    if degree >= 2:
        for i in range(n):
            result.append(row[i] ** 2)
        for i in range(n):
            for j in range(i + 1, n):
                result.append(row[i] * row[j])
    return result


### 步骤 2：从零实现类别编码


In [ ]:
def one_hot_encode(values):
    categories = sorted(set(values))
    cat_to_idx = {cat: i for i, cat in enumerate(categories)}
    n_cats = len(categories)

    encoded = []
    for v in values:
        row = [0] * n_cats
        row[cat_to_idx[v]] = 1
        encoded.append(row)

    return encoded, categories


def label_encode(values):
    categories = sorted(set(values))
    cat_to_int = {cat: i for i, cat in enumerate(categories)}
    return [cat_to_int[v] for v in values], cat_to_int


def target_encode(feature_values, target_values, smoothing=10):
    global_mean = sum(target_values) / len(target_values)

    category_stats = {}
    for feat, target in zip(feature_values, target_values):
        if feat not in category_stats:
            category_stats[feat] = {"sum": 0.0, "count": 0}
        category_stats[feat]["sum"] += target
        category_stats[feat]["count"] += 1

    encoding = {}
    for cat, stats in category_stats.items():
        cat_mean = stats["sum"] / stats["count"]
        weight = stats["count"] / (stats["count"] + smoothing)
        encoding[cat] = weight * cat_mean + (1 - weight) * global_mean

    return [encoding[v] for v in feature_values], encoding


### 步骤 3：从零实现文本特征


In [ ]:
def count_vectorize(documents):
    vocab = {}
    idx = 0
    for doc in documents:
        for word in doc.lower().split():
            if word not in vocab:
                vocab[word] = idx
                idx += 1

    vectors = []
    for doc in documents:
        vec = [0] * len(vocab)
        for word in doc.lower().split():
            vec[vocab[word]] += 1
        vectors.append(vec)

    return vectors, vocab


def tfidf(documents):
    n_docs = len(documents)

    vocab = {}
    idx = 0
    for doc in documents:
        for word in doc.lower().split():
            if word not in vocab:
                vocab[word] = idx
                idx += 1

    doc_freq = {}
    for doc in documents:
        seen = set()
        for word in doc.lower().split():
            if word not in seen:
                doc_freq[word] = doc_freq.get(word, 0) + 1
                seen.add(word)

    vectors = []
    for doc in documents:
        words = doc.lower().split()
        word_count = len(words)
        tf_map = {}
        for word in words:
            tf_map[word] = tf_map.get(word, 0) + 1

        vec = [0.0] * len(vocab)
        for word, count in tf_map.items():
            tf = count / word_count
            idf = math.log(n_docs / doc_freq[word])
            vec[vocab[word]] = tf * idf
        vectors.append(vec)

    return vectors, vocab


### 步骤 4：从零实现缺失值填补


In [ ]:
def impute_mean(values):
    present = [v for v in values if v is not None]
    if not present:
        return [0.0] * len(values), 0.0
    mean = sum(present) / len(present)
    return [v if v is not None else mean for v in values], mean


def impute_median(values):
    present = sorted(v for v in values if v is not None)
    if not present:
        return [0.0] * len(values), 0.0
    n = len(present)
    if n % 2 == 0:
        median = (present[n // 2 - 1] + present[n // 2]) / 2
    else:
        median = present[n // 2]
    return [v if v is not None else median for v in values], median


def impute_mode(values):
    present = [v for v in values if v is not None]
    if not present:
        return values, None
    counts = {}
    for v in present:
        counts[v] = counts.get(v, 0) + 1
    mode = max(counts, key=counts.get)
    return [v if v is not None else mode for v in values], mode


def add_missing_indicator(values):
    return [0 if v is not None else 1 for v in values]


### 步骤 5：从零实现特征选择


In [ ]:
def correlation(x, y):
    n = len(x)
    mean_x = sum(x) / n
    mean_y = sum(y) / n
    cov = sum((xi - mean_x) * (yi - mean_y) for xi, yi in zip(x, y)) / n
    std_x = math.sqrt(sum((xi - mean_x) ** 2 for xi in x) / n)
    std_y = math.sqrt(sum((yi - mean_y) ** 2 for yi in y) / n)
    if std_x == 0 or std_y == 0:
        return 0.0
    return cov / (std_x * std_y)


def mutual_information(feature, target, n_bins=10):
    feat_min = min(feature)
    feat_max = max(feature)
    bin_width = (feat_max - feat_min) / n_bins if feat_max != feat_min else 1.0
    feat_binned = [
        min(int((f - feat_min) / bin_width), n_bins - 1) for f in feature
    ]

    n = len(feature)
    target_classes = sorted(set(target))

    feat_bins = sorted(set(feat_binned))
    p_feat = {}
    for b in feat_bins:
        p_feat[b] = feat_binned.count(b) / n

    p_target = {}
    for t in target_classes:
        p_target[t] = target.count(t) / n

    mi = 0.0
    for b in feat_bins:
        for t in target_classes:
            joint_count = sum(
                1 for fb, tv in zip(feat_binned, target) if fb == b and tv == t
            )
            p_joint = joint_count / n
            if p_joint > 0:
                mi += p_joint * math.log(p_joint / (p_feat[b] * p_target[t]))

    return mi


def variance_threshold(features, threshold=0.01):
    n_features = len(features[0])
    n_samples = len(features)
    selected = []

    for j in range(n_features):
        col = [features[i][j] for i in range(n_samples)]
        mean = sum(col) / n_samples
        var = sum((v - mean) ** 2 for v in col) / n_samples
        if var >= threshold:
            selected.append(j)

    return selected


def remove_correlated(features, threshold=0.9):
    n_features = len(features[0])
    n_samples = len(features)

    to_remove = set()
    for i in range(n_features):
        if i in to_remove:
            continue
        col_i = [features[r][i] for r in range(n_samples)]
        for j in range(i + 1, n_features):
            if j in to_remove:
                continue
            col_j = [features[r][j] for r in range(n_samples)]
            corr = abs(correlation(col_i, col_j))
            if corr >= threshold:
                to_remove.add(j)

    return [i for i in range(n_features) if i not in to_remove]


### 步骤 6：完整流水线和演示


In [ ]:
import random


def make_housing_data(n=200, seed=42):
    random.seed(seed)
    data = []
    for _ in range(n):
        sqft = random.uniform(500, 5000)
        bedrooms = random.choice([1, 2, 3, 4, 5])
        age = random.uniform(0, 50)
        neighborhood = random.choice(["downtown", "suburbs", "rural"])
        has_pool = random.choice([True, False])

        sqft_with_missing = sqft if random.random() > 0.05 else None
        age_with_missing = age if random.random() > 0.08 else None

        price = (
            50 * sqft
            + 20000 * bedrooms
            - 1000 * age
            + (50000 if neighborhood == "downtown" else 10000 if neighborhood == "suburbs" else 0)
            + (15000 if has_pool else 0)
            + random.gauss(0, 20000)
        )

        data.append({
            "sqft": sqft_with_missing,
            "bedrooms": bedrooms,
            "age": age_with_missing,
            "neighborhood": neighborhood,
            "has_pool": has_pool,
            "price": price,
        })
    return data


if __name__ == "__main__":
    data = make_housing_data(200)

    print("=== Raw Data Sample ===")
    for row in data[:3]:
        print(f"  {row}")

    sqft_raw = [d["sqft"] for d in data]
    age_raw = [d["age"] for d in data]
    prices = [d["price"] for d in data]

    print("\n=== Missing Value Handling ===")
    sqft_missing = sum(1 for v in sqft_raw if v is None)
    age_missing = sum(1 for v in age_raw if v is None)
    print(f"  sqft missing: {sqft_missing}/{len(sqft_raw)}")
    print(f"  age missing: {age_missing}/{len(age_raw)}")

    sqft_indicator = add_missing_indicator(sqft_raw)
    age_indicator = add_missing_indicator(age_raw)
    sqft_imputed, sqft_fill = impute_median(sqft_raw)
    age_imputed, age_fill = impute_mean(age_raw)
    print(f"  sqft filled with median: {sqft_fill:.0f}")
    print(f"  age filled with mean: {age_fill:.1f}")

    print("\n=== Numerical Transforms ===")
    sqft_scaled = standardize(sqft_imputed)
    age_scaled = min_max_scale(age_imputed)
    sqft_log = log_transform(sqft_imputed)
    age_binned = bin_values(age_imputed, n_bins=5)
    print(f"  sqft standardized: mean={sum(sqft_scaled)/len(sqft_scaled):.4f}, std={math.sqrt(sum(v**2 for v in sqft_scaled)/len(sqft_scaled)):.4f}")
    print(f"  age min-max: [{min(age_scaled):.2f}, {max(age_scaled):.2f}]")
    print(f"  age bins: {sorted(set(age_binned))}")

    print("\n=== Categorical Encoding ===")
    neighborhoods = [d["neighborhood"] for d in data]

    ohe, ohe_cats = one_hot_encode(neighborhoods)
    print(f"  One-hot categories: {ohe_cats}")
    print(f"  Sample encoding: {neighborhoods[0]} -> {ohe[0]}")

    le, le_map = label_encode(neighborhoods)
    print(f"  Label encoding map: {le_map}")

    te, te_map = target_encode(neighborhoods, prices, smoothing=10)
    print(f"  Target encoding: {({k: round(v) for k, v in te_map.items()})}")

    print("\n=== Text Features ===")
    descriptions = [
        "large modern house with pool",
        "small cozy cottage near downtown",
        "spacious family home with large yard",
        "modern apartment downtown with view",
        "rustic cabin in rural area",
    ]
    cv, cv_vocab = count_vectorize(descriptions)
    print(f"  Vocabulary size: {len(cv_vocab)}")
    print(f"  Doc 0 non-zero features: {sum(1 for v in cv[0] if v > 0)}")

    tf, tf_vocab = tfidf(descriptions)
    print(f"  TF-IDF vocabulary size: {len(tf_vocab)}")
    top_words = sorted(tf_vocab.keys(), key=lambda w: tf[0][tf_vocab[w]], reverse=True)[:3]
    print(f"  Doc 0 top TF-IDF words: {top_words}")

    print("\n=== Polynomial Features ===")
    sample_row = [sqft_scaled[0], age_scaled[0]]
    poly = polynomial_features(sample_row, degree=2)
    print(f"  Input: {[round(v, 4) for v in sample_row]}")
    print(f"  Polynomial: {[round(v, 4) for v in poly]}")
    print(f"  Features: [x1, x2, x1^2, x2^2, x1*x2]")

    print("\n=== Feature Selection ===")
    feature_matrix = [
        [sqft_scaled[i], age_scaled[i], float(sqft_indicator[i]), float(age_indicator[i])]
        + ohe[i]
        for i in range(len(data))
    ]

    print(f"  Total features: {len(feature_matrix[0])}")

    surviving_var = variance_threshold(feature_matrix, threshold=0.01)
    print(f"  After variance threshold (0.01): {len(surviving_var)} features kept")

    surviving_corr = remove_correlated(feature_matrix, threshold=0.9)
    print(f"  After correlation filter (0.9): {len(surviving_corr)} features kept")

    binary_prices = [1 if p > sum(prices) / len(prices) else 0 for p in prices]
    print("\n  Mutual information with target:")
    feature_names = ["sqft", "age", "sqft_missing", "age_missing"] + [f"neigh_{c}" for c in ohe_cats]
    for j in range(len(feature_matrix[0])):
        col = [feature_matrix[i][j] for i in range(len(feature_matrix))]
        mi = mutual_information(col, binary_prices, n_bins=10)
        print(f"    {feature_names[j]}: MI={mi:.4f}")

    print("\n  Correlation with price:")
    for j in range(len(feature_matrix[0])):
        col = [feature_matrix[i][j] for i in range(len(feature_matrix))]
        corr = correlation(col, prices)
        print(f"    {feature_names[j]}: r={corr:.4f}")


## 实际使用

使用 scikit-learn 时，这些变换可以组合成 pipelines：


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import mutual_info_classif, VarianceThreshold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("encoder", OneHotEncoder(sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, ["sqft", "age"]),
    ("cat", categorical_pipe, ["neighborhood"]),
])


从零实现的版本展示了每个变换内部到底发生了什么。库版本会增加边界情况处理、稀疏矩阵支持和 pipeline 组合能力，但数学是一样的。

## 交付成果

本课产出：
- `outputs/prompt-feature-engineer.md` - 一个用于从原始数据系统化设计特征的 prompt

## 练习

1. 在数值变换中加入 robust scaling（使用中位数和四分位距，而不是均值和标准差）。在有极端离群值的数据上，把它和 standard scaling 做比较。
2. 实现 leave-one-out target encoding：对每一行，计算目标均值时排除该行自己的目标值。说明它如何相比朴素 target encoding 减少过拟合。
3. 构建一个自动化特征选择流水线，组合 variance threshold、correlation filtering 和 mutual information ranking。把它应用到房屋数据集上，并比较使用全部特征与使用选中特征时的模型表现（使用简单的 linear regression）。

## 关键术语

| 术语 | 常见说法 | 实际含义 |
|------|----------------|----------------------|
| Feature engineering | “做新列” | 把原始数据转换成能向模型暴露模式的表示 |
| Standardization | “让它变成正态” | 减去均值并除以标准差，使特征具有 mean=0 和 std=1 |
| One-hot encoding | “做 dummy variables” | 为每个类别创建一个二进制列，每一行恰好有一列为 1 |
| Target encoding | “用答案来编码” | 用该类别的目标平均值替换每个类别，并使用 smoothing 防止过拟合 |
| TF-IDF | “高级词频” | Term Frequency 乘以 Inverse Document Frequency：按词在语料库中有多独特来加权 |
| Imputation | “填空” | 用估计值（均值、中位数、众数或模型预测值）替换缺失值 |
| Feature selection | “扔掉坏列” | 移除增加噪声或冗余的特征，只保留包含目标信号的特征 |
| Mutual information | “一件事能告诉你另一件事多少” | 观察变量 X 后，对变量 Y 的不确定性减少量的一种度量 |
| Data leakage | “不小心作弊” | 训练时使用了预测时不可获得的信息，导致结果虚假乐观 |

## 延伸阅读

- [Feature Engineering and Selection (Max Kuhn & Kjell Johnson)](http://www.feat.engineering/) - 一本免费的在线书，覆盖特征工程的完整版图
- [scikit-learn Preprocessing Guide](https://scikit-learn.org/stable/modules/preprocessing.html) - 所有标准变换的实用参考
- [Target Encoding Done Right (Micci-Barreca, 2001)](https://dl.acm.org/doi/10.1145/507533.507538) - 关于带 smoothing 的 target encoding 的原始论文
